> **STAGE-2 SUPERSEDED NOTICE:** This Stage-1 notebook is superseded by **[`08_confirmatory_operating_point_matching.ipynb`](file://./notebooks/08_confirmatory_operating_point_matching.ipynb)** (30 confirmatory runs on held-out seeds `201..205`) and **[`09_actual_ace_intervention.ipynb`](file://./notebooks/09_actual_ace_intervention.ipynb)** (which implements the actual contrastive **Attractor-Contrast-Escape [ACE]** algorithm of Zhang et al., 2026 rather than the online `Rank-1 Step-Delta SVD` heuristic).

# Notebook 02 — Phases 2 & 7: Colab GPU Runner for Root-Matched $\gamma^*$ Pairs & Self-Conditioning Direction Attenuation (`ACE` Hook)

**Status:**
- **Local Audit / Dry-Run:** Verified (`EXIT_CODE=0`).
- **Full GPU Generation:** **PENDING COLAB A100 EXECUTION** (~1.1h for Part A root-matched pairs + ~0.9h for Part B self-conditioning direction attenuation).
- **Scientific Integrity Guarantee:** No fabricated or pre-populated GPU outputs are stored in this notebook. All outputs write to unique run IDs under `outputs/runs/p9_followup/`.

---

### What This Notebook Executes on Colab GPU:
1. **Part A (Phase 2 Exact Continuous Root-Matched Operating Points):**
   Runs $n=5$ independent seeds (`101..105`) at the continuous root-matched $\gamma^*$ values identified in `Notebook 01`:
   - `Pair R1 (ELF-B)`: $(\text{SC-CFG}=1.0, \gamma^*=1.349)$ vs. $(\text{SC-CFG}=2.0, \gamma=1.000)$ (predicted $\Delta\text{Gen-PPL}=-0.04\%, \Delta H_{\text{ps}}=+0.0002$ nats).
   - `Pair R2 (ELF-B, exact S2 root)`: $(\text{SC-CFG}=1.0, \gamma^*=1.548)$ vs. $(\text{SC-CFG}=3.0, \gamma=1.000)$ (predicted $\Delta\text{Gen-PPL}=+0.03\%, \Delta H_{\text{ps}}=-0.0005$ nats).
   - `Pair R3 (ELF-B, exact Refined-2 root)`: $(\text{SC-CFG}=1.0, \gamma^*=1.652)$ vs. $(\text{SC-CFG}=4.0, \gamma=1.000)$ (predicted $\Delta\text{Gen-PPL}=+0.26\%, \Delta H_{\text{ps}}=-0.0028$ nats).
2. **Part B (Phase 7 — Zhang et al. 2026 Dominant Self-Conditioning Direction Attenuation Hook):**
   Uses the `hook(i, t, t_next, t_eff, z, x_pred)` interface in `elfx/sample.py` (line 175) to compute the token-wise self-conditioning displacement $\mathbf{D}^{(i)} = \mathbf{x}_{\text{pred}}^{(i)} - \mathbf{x}_{\text{pred}}^{(i-1)}$ at each ODE/SDE step and attenuate its top singular direction $\hat{\mathbf{v}}_1$ by factor $\alpha_{\text{ACE}} \in \{0.0, 0.5, 1.0\}$ across matched pairs `S1` (`ELF-B`) and `M1` (`ELF-M`).

In [1]:
# Cell 1: Dry-Run Manifest & Hook Verification (Runs on CPU or GPU without downloading 3GB weights)
import math
import torch

ROOT_MATCHED_MANIFEST = [
    # Pair R1: (SC=1.0, g=1.349) vs (SC=2.0, g=1.0)
    {"pair": "R1_ELFB", "model": "ELF-B", "sc_cfg": 1.0, "gamma": 1.349, "seeds": [101, 102, 103, 104, 105], "steps": 32},
    {"pair": "R1_ELFB", "model": "ELF-B", "sc_cfg": 2.0, "gamma": 1.000, "seeds": [103, 104, 105], "steps": 32},  # 101, 102 already on disk
    # Pair R2 (Exact S2 Root): (SC=1.0, g=1.548) vs (SC=3.0, g=1.0)
    {"pair": "R2_ELFB", "model": "ELF-B", "sc_cfg": 1.0, "gamma": 1.548, "seeds": [101, 102, 103, 104, 105], "steps": 32},
    {"pair": "R2_ELFB", "model": "ELF-B", "sc_cfg": 3.0, "gamma": 1.000, "seeds": [103, 104, 105], "steps": 32},  # 101, 102 already on disk
    # Pair R3 (Exact Refined-2 Root): (SC=1.0, g=1.652) vs (SC=4.0, g=1.0)
    {"pair": "R3_ELFB", "model": "ELF-B", "sc_cfg": 1.0, "gamma": 1.652, "seeds": [101, 102, 103, 104, 105], "steps": 32},
]

def make_self_cond_direction_attenuation_hook(alpha_ace: float = 0.5):
    """
    Operationalizes dominant self-conditioning direction attenuation (inspired by Zhang et al., 2026)
    using elfx/sample.py's per-step hook(i, t, t_next, t_eff, z, x_pred) -> (z, x_pred).
    At each step i >= 1, computes the token-wise self-conditioning update Delta_x = x_pred - x_pred_prev,
    extracts the top right singular vector v_1 across the L tokens for each sample in the batch,
    and attenuates that rank-1 directional drift by alpha_ace in [0, 1].
    """
    state = {"x_pred_prev": None}

    @torch.no_grad()
    def hook(i, t, t_next, t_eff, z, x_pred):
        if i == 0 or state["x_pred_prev"] is None or alpha_ace <= 0.0:
            state["x_pred_prev"] = x_pred.detach().clone()
            return z, x_pred
        delta = (x_pred - state["x_pred_prev"]).float()  # (B, L, D)
        # Power iteration (2 steps) for top right singular vector v_1 of shape (B, D, 1)
        v = delta.mean(dim=1, keepdim=True).transpose(1, 2)  # (B, D, 1)
        v = v / v.norm(dim=1, keepdim=True).clamp(min=1e-8)
        for _ in range(2):
            u = torch.bmm(delta, v)  # (B, L, 1)
            v = torch.bmm(delta.transpose(1, 2), u)  # (B, D, 1)
            v = v / v.norm(dim=1, keepdim=True).clamp(min=1e-8)
        proj = torch.bmm(torch.bmm(delta, v), v.transpose(1, 2)).to(x_pred.dtype)
        x_pred_attenuated = x_pred - alpha_ace * proj
        state["x_pred_prev"] = x_pred_attenuated.detach().clone()
        return z, x_pred_attenuated

    return hook

# Unit-test the hook on a synthetic (B=4, L=128, D=512) latent trajectory
hook_fn = make_self_cond_direction_attenuation_hook(alpha_ace=0.5)
z_dummy = torch.randn(4, 128, 512)
x0 = torch.randn(4, 128, 512)
# Simulate a rank-1 drift + isotropic change
drift_dir = torch.randn(4, 1, 512)
x1 = x0 + 2.0 * drift_dir + 0.1 * torch.randn(4, 128, 512)
_, x0_out = hook_fn(0, 0.0, 0.03, 0.0, z_dummy, x0)
_, x1_out = hook_fn(1, 0.03, 0.06, 0.03, z_dummy, x1)
raw_drift_norm = (x1 - x0).mean(dim=1).norm(dim=-1).mean().item()
att_drift_norm = (x1_out - x0_out).mean(dim=1).norm(dim=-1).mean().item()
total_pending_runs = sum(len(x["seeds"]) for x in ROOT_MATCHED_MANIFEST)
print(f"[Audit OK] Total pending Part A root-matched runs: {total_pending_runs} runs (~28 mins on Colab A100)")
print(f"[Hook Unit Test OK] Token-mean drift norm reduced from {raw_drift_norm:.2f} -> {att_drift_norm:.2f} (expected 50% attenuation = {0.5*raw_drift_norm:.2f})")

[Audit OK] Total pending Part A root-matched runs: 21 runs (~28 mins on Colab A100)
[Hook Unit Test OK] Token-mean drift norm reduced from 44.36 -> 22.18 (expected 50% attenuation = 22.18)


In [ ]:
# Cell 2: Google Colab GPU Execution Cell (Run on Colab A100/L4 GPU)
# Guarded by RUN_GPU_EXPERIMENTS = False by default so local inspection never triggers accidental multi-hour runs.
RUN_GPU_EXPERIMENTS = False

if RUN_GPU_EXPERIMENTS:
    import json, time
    from pathlib import Path
    from elfx.load import load_elf, load_gpt2_large
    from elfx.sample import sample_corpus
    from elfx.metrics import score_gen_ppl_and_entropy, score_rep_and_distinct
    from analysis.rescore_existing_texts import compute_extended_metrics_for_texts

    out_root = Path("outputs/runs/p9_followup")
    out_root.mkdir(parents=True, exist_ok=True)
    bundle = load_elf("ELF-B")
    gpt2, gpt2_tok = load_gpt2_large()

    # 1. Execute Part A: Continuous root-matched pairs
    for job in ROOT_MATCHED_MANIFEST:
        for seed in job["seeds"]:
            run_id = f"p9_{job['pair']}_sc{str(job['sc_cfg']).replace('.','p')}_g{str(job['gamma']).replace('.','p')}_sd{seed}"
            run_dir = out_root / run_id
            if (run_dir / "summary.json").exists():
                print(f"[SKIP] {run_id} already exists")
                continue
            run_dir.mkdir(parents=True, exist_ok=True)
            res = sample_corpus(
                bundle, n_samples=1000, batch_size=128, steps=job["steps"],
                schedule="pow2", sc_cfg=job["sc_cfg"], gamma=job["gamma"], seed=seed
            )
            ppl_res = score_gen_ppl_and_entropy(res["texts"], gpt2, gpt2_tok)
            lex_res = score_rep_and_distinct(res["texts"], bundle.tokenizer)
            summary = {**job, "seed": seed, "run_id": run_id, **ppl_res, **lex_res, "wall_s": res["wall_s"], "nfe_elf": res["nfe_elf"]}
            (run_dir / "texts.json").write_text(json.dumps(res["texts"], indent=2))
            (run_dir / "summary.json").write_text(json.dumps(summary, indent=2))
            print(f"[DONE] {run_id}: Gen-PPL={summary['gen_ppl']:.2f}, H_ps={summary['entropy']:.4f}, rep4={summary['rep4']:.4f}")

    # 2. Execute Part B: Self-Conditioning Dominant Direction Attenuation (alpha_ACE in {0.0, 0.5, 1.0}) on S1_ELFB
    for alpha_ace in [0.0, 0.5, 1.0]:
        for sc_cfg, gamma in [(1.0, 2.0), (4.0, 1.5)]:
            for seed in [101, 102, 103]:
                run_id = f"p9_ace_a{str(alpha_ace).replace('.','p')}_sc{int(sc_cfg)}_g{str(gamma).replace('.','p')}_sd{seed}"
                run_dir = out_root / run_id
                if (run_dir / "summary.json").exists():
                    continue
                run_dir.mkdir(parents=True, exist_ok=True)
                hook_fn = make_self_cond_direction_attenuation_hook(alpha_ace=alpha_ace)
                res = sample_corpus(
                    bundle, n_samples=1000, batch_size=128, steps=32,
                    schedule="pow2", sc_cfg=sc_cfg, gamma=gamma, seed=seed, hook=hook_fn
                )
                ppl_res = score_gen_ppl_and_entropy(res["texts"], gpt2, gpt2_tok)
                lex_res = score_rep_and_distinct(res["texts"], bundle.tokenizer)
                summary = {"alpha_ace": alpha_ace, "sc_cfg": sc_cfg, "gamma": gamma, "seed": seed, **ppl_res, **lex_res, "wall_s": res["wall_s"]}
                (run_dir / "texts.json").write_text(json.dumps(res["texts"], indent=2))
                (run_dir / "summary.json").write_text(json.dumps(summary, indent=2))
                print(f"[DONE ACE] {run_id}: Gen-PPL={summary['gen_ppl']:.2f}, H_ps={summary['entropy']:.4f}, rep4={summary['rep4']:.4f}")